# OpenPlaque RCA Proximal Automatic Refinement

No sliders, widgets, clicks, or manual coordinates. This notebook loads source CCTA series **7**, detects the ascending aorta, restricts the search to the patient-right/anterior aortic-root annulus, and ranks only persistent proximal-RCA-like structures. It then renders a dense static sequence around the strongest candidate plus thin-slab MIPs.

Use **Runtime → Run all**. Google Drive mounts first.


In [ ]:
# ALWAYS FIRST: mount Google Drive.
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive mounted.')


In [ ]:
!rm -rf /content/OpenPlaque
!git clone -q --branch rca-centerline-from-main --single-branch https://github.com/pazzani/OpenPlaque.git /content/OpenPlaque
%pip -q install pydicom SimpleITK scipy scikit-image matplotlib pandas

import os, sys, time, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SRC = Path('/content/OpenPlaque/src')
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
from openplaque.study import OpenPlaqueStudy
from openplaque.rca_root_candidates import find_rca_ostium_candidates
print('OpenPlaque source:', SRC)


## Load source CCTA series 7


In [ ]:
ROOT = Path('/content/drive/MyDrive/OpenPlaque')
DRIVE_ZIP = ROOT / 'Full_DICOM.zip'
LOCAL_ZIP = Path('/content/Full_DICOM.zip')
EXTRACT_ROOT = '/content/full_dicom_rca_proximal_refine'
SOURCE_SERIES = 7
if not DRIVE_ZIP.exists():
    raise FileNotFoundError(f'Missing {DRIVE_ZIP}')
if not LOCAL_ZIP.exists() or LOCAL_ZIP.stat().st_size != DRIVE_ZIP.stat().st_size:
    print('Copying Full_DICOM.zip to local disk...', flush=True)
    shutil.copyfile(DRIVE_ZIP, LOCAL_ZIP)
else:
    print('Local ZIP already staged.')
shutil.rmtree(EXTRACT_ROOT, ignore_errors=True)
t=time.time()
study = OpenPlaqueStudy(str(LOCAL_ZIP), extract_root=EXTRACT_ROOT)
print(f'DICOM scan: {time.time()-t:.1f}s; {len(study.series)} series')
source_img, source, _ = study.load_series(SOURCE_SERIES)
spacing = source_img.GetSpacing()
print('Series 7 shape zyx:', source.shape)
print('Spacing xyz mm:', spacing)


## Anatomy-constrained proximal RCA candidates


In [ ]:
print('Searching only around the ascending-aortic root...', flush=True)
t=time.time()
candidates, root_window = find_rca_ostium_candidates(source, spacing, n=6)
print(f'Finished in {time.time()-t:.1f}s. Root window z={root_window[0]}..{root_window[1]-1}')
if not candidates:
    raise RuntimeError('No anatomy-constrained RCA candidates found')
df = pd.DataFrame([dict(
    candidate=f'R{i+1}', z=c.z, x=c.x, y=c.y, score=c.score, HU=c.hu,
    vesselness=c.vesselness, radius_mm=c.local_radius_mm, support_slices=c.support_slices,
    aorta_x=c.aorta_x, aorta_y=c.aorta_y, aorta_radius_mm=c.aorta_radius_mm
) for i,c in enumerate(candidates)])
display(df)


## Top hypotheses with full anatomical context
Each candidate is shown on the full axial image and in a local aortic-root crop. The circle approximates the detected ascending-aortic boundary; the crosshair marks the RCA hypothesis.


In [ ]:
OUTDIR = ROOT / 'RCA_Proximal_Refinement'
OUTDIR.mkdir(parents=True, exist_ok=True)
fig, axes = plt.subplots(len(candidates), 2, figsize=(13, 4*len(candidates)))
if len(candidates)==1:
    axes=np.asarray([axes])
for i,c in enumerate(candidates):
    z,y,x=c.z,c.y,c.x
    axes[i,0].imshow(source[z], cmap='gray', vmin=-200, vmax=800)
    axes[i,0].scatter([x],[y],s=70,facecolors='none',edgecolors='cyan')
    axes[i,0].set_title(f'R{i+1} full context  z={z}')
    axes[i,0].axis('off')
    rpix = c.aorta_radius_mm / np.sqrt(spacing[0]*spacing[1])
    circ=plt.Circle((c.aorta_x,c.aorta_y),rpix,fill=False,linewidth=1.2)
    axes[i,0].add_patch(circ)
    r=110
    cy,cx=int(round(c.aorta_y)),int(round(c.aorta_x))
    y0,y1=max(0,cy-r),min(source.shape[1],cy+r+1)
    x0,x1=max(0,cx-r),min(source.shape[2],cx+r+1)
    axes[i,1].imshow(source[z,y0:y1,x0:x1], cmap='gray', vmin=-200, vmax=800)
    axes[i,1].scatter([x-x0],[y-y0],s=90,facecolors='none',edgecolors='cyan')
    axes[i,1].set_title(f'R{i+1} root crop  HU={c.hu:.0f}  support={c.support_slices}')
    axes[i,1].axis('off')
plt.tight_layout()
p=OUTDIR/'RCA_top_root_hypotheses.png'
fig.savefig(p,dpi=180,bbox_inches='tight')
plt.show(); plt.close(fig)
print('Saved:',p)


## Dense sequence around the strongest candidate
This is the most informative view for deciding whether the structure truly emerges from the right coronary sinus.


In [ ]:
best=candidates[0]
dz=max(1,int(round(1.2/spacing[2])))
zs=np.arange(best.z-5*dz,best.z+6*dz,dz,dtype=int)
zs=zs[(zs>=0)&(zs<source.shape[0])]
cy,cx=int(round(best.aorta_y)),int(round(best.aorta_x))
r=125
y0,y1=max(0,cy-r),min(source.shape[1],cy+r+1)
x0,x1=max(0,cx-r),min(source.shape[2],cx+r+1)
fig,axes=plt.subplots(3,4,figsize=(16,12))
for ax in axes.ravel(): ax.axis('off')
for ax,z in zip(axes.ravel(),zs):
    ax.imshow(source[z,y0:y1,x0:x1],cmap='gray',vmin=-200,vmax=800)
    ax.set_title(f'z={z}  ({(z-best.z)*spacing[2]:+.1f} mm from R1)')
    ax.axis('off')
plt.tight_layout()
p=OUTDIR/'RCA_R1_dense_root_sequence.png'
fig.savefig(p,dpi=200,bbox_inches='tight')
plt.show(); plt.close(fig)
print('Saved:',p)


## Thin-slab MIPs through the root
Thin slabs often make the short proximal RCA segment easier to distinguish from isolated bright dots.


In [ ]:
fig,axes=plt.subplots(1,3,figsize=(15,5))
for ax,half_mm in zip(axes,[1.5,3.0,5.0]):
    h=max(1,int(round(half_mm/spacing[2])))
    za=max(0,best.z-h); zb=min(source.shape[0],best.z+h+1)
    slab=np.max(source[za:zb,y0:y1,x0:x1],axis=0)
    ax.imshow(slab,cmap='gray',vmin=-100,vmax=800)
    ax.set_title(f'R1 thin-slab MIP ±{half_mm:.1f} mm')
    ax.axis('off')
plt.tight_layout()
p=OUTDIR/'RCA_R1_thin_slab_MIPs.png'
fig.savefig(p,dpi=200,bbox_inches='tight')
plt.show(); plt.close(fig)
print('Saved:',p)
print('\nDONE. Upload RCA_top_root_hypotheses.png, RCA_R1_dense_root_sequence.png, and RCA_R1_thin_slab_MIPs.png.')
